# 20 — Ordering policy bakeoff (window SLA controllers)

Compare **damped survival-weighted** (`sw`), **window SLA fast path** (`sla_pb`),
**window SLA Monte Carlo oracle** (`sla_mc`), and **rollout** on shared seeds
using the Rust kernel alpha-tune ladder (T-164 / ADR 0151).

Set `SMOKE=True` for one seed and three scored days.

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")

from blueberries_voi.backend import rust_available, warn_fallback_once
from blueberries_voi.experiments.policy_bakeoff_viz import write_runtime_frontier_figure
from blueberries_voi.sim.alpha_tune import evaluate_alpha_episode_outcomes
from blueberries_voi.sim.service import service_metrics_from_steps
from blueberries_voi.sim.shipments import smoke_cool_shipments

SMOKE = False
SEEDS = (42,) if SMOKE else (42, 7, 101)
N_BURN = 2 if SMOKE else 7
N_SCORE = 3 if SMOKE else 14
ALPHA = 0.9
RHO = 0.8
ARMS = ("sw", "sla_pb", "sla_mc", "rollout")
ARM_IDX = {name: float(i) for i, name in enumerate(ARMS)}
OUT_DIR = Path("notebooks/outputs/nb20_policy_bakeoff")

warn_fallback_once()
if not rust_available():
    raise RuntimeError(
        "Rust backend required. From repo root: maturin develop --release -m crates/voi_py/Cargo.toml"
    )
print(f"SMOKE={SMOKE} seeds={SEEDS} arms={ARMS}")

In [ ]:
rows: list[dict[str, float]] = []
ships = smoke_cool_shipments()

for arm in ARMS:
    rollout_h = 7 if arm == "rollout" else 2
    n_paths = 8 if arm == "rollout" else 1
    radius = 2 if arm == "rollout" else 1
    for seed in SEEDS:
        t0 = time.perf_counter()
        out = evaluate_alpha_episode_outcomes(
            arm,
            ALPHA,
            seed,
            rho=RHO,
            shipments=ships,
            n_burn=N_BURN,
            n_score=N_SCORE,
            rollout_h=rollout_h,
            n_rollout_paths=n_paths,
            candidate_case_radius=radius,
        )
        elapsed = time.perf_counter() - t0
        fill = 1.0 - (out.total_lost_sales / max(1, N_SCORE * 20))
        rows.append(
            {
                "arm_idx": ARM_IDX[arm],
                "profit": out.profit,
                "waste": float(out.total_waste),
                "lost": float(out.total_lost_sales),
                "seconds": elapsed,
                "fill_proxy": fill,
            }
        )
        print(
            f"{arm:8} seed={seed} profit={out.profit:.1f} "
            f"waste={out.total_waste} lost={out.total_lost_sales} t={elapsed:.2f}s"
        )

write_runtime_frontier_figure(OUT_DIR / "runtime_frontier.json", rows)
print(f"Wrote {OUT_DIR / 'runtime_frontier.json'}")

In [ ]:
# Service metrics helper (fill rate / day no-stockout) on scored step tallies.
from dataclasses import dataclass

@dataclass
class _Step:
    demand: int
    sales_total: int

demo = service_metrics_from_steps([_Step(10, 9), _Step(8, 8), _Step(12, 10)])
print(
    f"fill_rate={demo.fill_rate:.3f} "
    f"day_no_stockout_rate={demo.day_no_stockout_rate:.3f} days={demo.scored_days}"
)